# NB03b — Synthetic Generation (FLUX Zero-shot)

สร้าง synthetic image สำหรับ condition C ของ NB04 ใช้ FLUX.2-klein base ตรงๆ ไม่สวม LoRA prompt ใช้ binomial name ของ species gen ภาพที่ res 512 ตามสัดส่วน organ จริงของ species นั้น budget ต่อ species = need = T - n_train(real) เท่ากับ NB03a เพื่อให้ NB04 fill ถึง T พอดี

output: append เข้า data/metadata/synthetic_manifest.csv (path, species_id, source=flux_zeroshot) โดยไม่แตะแถวของ lora_boost ที่ NB03a เขียนไว้

## Section 1) Setup

ตั้ง path โหลด species.csv กรอง 15 rare species + เก็บ binomial (2 คำแรกของ scientific_name) ไว้ใช้แทน trigger ใน prompt source ล็อกเป็น flux_zeroshot (03b = condition C) BASE_MODEL ดึงจาก nb02_handoff.json ให้ตรงกับ NB03a

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('/workspace/lora-boost')
DATA = ROOT / 'data'
META = DATA / 'metadata'
IMG_ROOT = DATA / 'raw' / 'plantnet_300K' / 'images'
SYNTH_ROOT = DATA / 'synthetic'
RESULTS = ROOT / 'results' / 'nb03b'

SOURCE = 'flux_zeroshot'                    # 03b = condition C
SYNTH_DIR = SYNTH_ROOT / SOURCE             # generate ลง data/synthetic/flux_zeroshot/<id>/
for p in (SYNTH_DIR, RESULTS):
    p.mkdir(parents=True, exist_ok=True)

handoff = json.load(open(META / 'nb02_handoff.json'))
BASE_MODEL = handoff['base_model']

species_df = pd.read_csv(META / 'species.csv', dtype={'species_id': str})
rare_df = species_df[species_df.is_rare].copy().reset_index(drop=True)
rare_df['binomial'] = rare_df['scientific_name'].str.split().str[:2].str.join(' ')

CAPTION_TMPL = 'a photo of the {organ} of a {name} plant'

print(f'source      :\t{SOURCE}')
print(f'base model  :\t{BASE_MODEL}')
print(f'caption     :\t{CAPTION_TMPL}')
print(f'species     :\t{len(rare_df)}')
print()
print(rare_df[['species_id', 'binomial']].to_string(index=False))

source      :	flux_zeroshot
base model  :	black-forest-labs/FLUX.2-klein-base-4B
caption     :	a photo of the {organ} of a {name} plant
species     :	15

species_id                     binomial
   1416509        Nepenthes vieillardii
   1419807           Vanilla planifolia
   1376703           Petiveria alliacea
   1409195            Pereskia aculeata
   1408557             Pongamia pinnata
   1420787               Alocasia odora
   1406486             Kigelia africana
   1408490         Pereskia grandifolia
   1385700        Barringtonia asiatica
   1356076         Tradescantia zebrina
   1397598              Cobaea scandens
   1618661 Dendrobium victoriae-reginae
   1384499          Lagenaria siceraria
   1406863              Acalypha indica
   1392654     Gomphocarpus physocarpus


## Section 2) Config & generation plan

cfg เดียวกับ NB03a เพื่อให้ condition C เทียบกับ D ตรงๆ ที่ปริมาณรูปและสัดส่วน organ เท่ากัน budget ต่อ species = T - n_train(real) คำนวณจาก splits.csv ฝั่ง rare train

In [2]:
import numpy as np

cfg = {
    'resolution': 512,
    'steps': 28,
    'guidance': 4.0,
    'seed': 42,
    'gen_batch': 1,
    'T': 200,
}

for k, v in cfg.items():
    print(f'{k:12}:\t{v}')

resolution  :	512
steps       :	28
guidance    :	4.0
seed        :	42
gen_batch   :	1
T           :	200


In [3]:
splits_df = pd.read_csv(META / 'splits.csv', dtype={'species_id': str})
rare_train = splits_df[(splits_df.is_rare) & (splits_df.split == 'train')]

rows = []
for r in rare_df.itertuples():
    sid = r.species_id
    g = rare_train[rare_train.species_id == sid]
    n_flower = int((g.organ == 'flower').sum())
    n_leaf = int((g.organ == 'leaf').sum())
    n_train = n_flower + n_leaf
    budget = max(0, cfg['T'] - n_train)
    fl_frac = n_flower / n_train if n_train else 0.0
    gen_flower = round(budget * fl_frac)
    gen_leaf = budget - gen_flower
    rows.append({
        'species_id': sid,
        'name': r.binomial,
        'n_train': n_train,
        'budget': budget,
        'gen_flower': gen_flower,
        'gen_leaf': gen_leaf,
    })

plan_df = pd.DataFrame(rows).sort_values('budget', ascending=False).reset_index(drop=True)

print(plan_df.to_string(index=False))
print(f'\ntotal budget :\t{plan_df.budget.sum():,} images')

species_id                         name  n_train  budget  gen_flower  gen_leaf
   1416509        Nepenthes vieillardii       20     180          72       108
   1419807           Vanilla planifolia       23     177          46       131
   1376703           Petiveria alliacea       28     172          43       129
   1409195            Pereskia aculeata       33     167          40       127
   1408557             Pongamia pinnata       36     164          50       114
   1406486             Kigelia africana       37     163          70        93
   1420787               Alocasia odora       38     162          30       132
   1408490         Pereskia grandifolia       40     160         116        44
   1385700        Barringtonia asiatica       47     153          85        68
   1356076         Tradescantia zebrina       49     151          22       129
   1397598              Cobaea scandens       49     151         117        34
   1618661 Dendrobium victoriae-reginae       51    

## Section 3) Generation

โหลด FLUX.2-klein ครั้งเดียว ไม่สวม LoRA generate ทีละ species ตาม budget แยก flower/leaf ตามแผน prompt ใช้ binomial name ของ species seed ไล่ไม่ซ้ำ เซฟรูปลง data/synthetic/flux_zeroshot/<id>/ พร้อมจด gen_manifest (species_id, organ, seed, path) ทุกใบ idempotent: species ไหนมีครบ budget แล้วข้าม pod หลุดมารันต่อได้

In [4]:
import torch
from diffusers import Flux2KleinPipeline

if 'pipe' not in dir():
    pipe = Flux2KleinPipeline.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16).to('cuda')
print(f'pipe loaded :\t{torch.cuda.memory_allocated() / 1e9:.1f} GB')

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

pipe loaded :	16.0 GB


In [5]:
GEN_MANIFEST = RESULTS / 'gen_manifest.csv'
GEN_COLS = ['species_id', 'organ', 'seed', 'path']

def load_gen_manifest():
    if GEN_MANIFEST.exists():
        return pd.read_csv(GEN_MANIFEST, dtype={'species_id': str, 'seed': int})
    return pd.DataFrame(columns=GEN_COLS)

@torch.no_grad()
def generate_species(sid, name, gen_flower, gen_leaf, done_seeds):
    out_dir = SYNTH_DIR / sid
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    seed = 0
    for organ, n_target in (('flower', gen_flower), ('leaf', gen_leaf)):
        prompt = CAPTION_TMPL.format(organ=organ, name=name)
        made = 0
        while made < n_target:
            bs = min(cfg['gen_batch'], n_target - made)
            seeds = []
            while len(seeds) < bs:
                if (sid, seed) not in done_seeds:
                    seeds.append(seed)
                seed += 1
            gens = [torch.Generator('cuda').manual_seed(s) for s in seeds]
            imgs = pipe(prompt=prompt, num_inference_steps=cfg['steps'],
                        guidance_scale=cfg['guidance'],
                        height=cfg['resolution'], width=cfg['resolution'],
                        num_images_per_prompt=bs, generator=gens).images
            for s, img in zip(seeds, imgs):
                fn = f'{sid}_{organ}_{s:05d}.png'
                img.save(out_dir / fn)
                rows.append({'species_id': sid, 'organ': organ, 'seed': s, 'path': fn})
            made += bs
    return rows

In [7]:
import time

for r in plan_df.itertuples():
    sid = r.species_id
    name = rare_df.loc[rare_df.species_id == sid, 'binomial'].iloc[0]

    man = load_gen_manifest()
    have = man[man.species_id == sid]
    if len(have) >= r.budget:
        print(f'{sid} {r.name:28} skip (have {len(have)}/{r.budget})')
        continue

    done_seeds = set(zip(have.species_id, have.seed))
    t0 = time.time()
    rows = generate_species(sid, name, r.gen_flower, r.gen_leaf, done_seeds)

    out = pd.concat([man, pd.DataFrame(rows)], ignore_index=True)
    out.to_csv(GEN_MANIFEST, index=False)
    print(f'{sid} {r.name:28} +{len(rows):4d} -> {len(out[out.species_id == sid]):4d}/{r.budget}  {(time.time() - t0) / 60:.1f} min')

man = load_gen_manifest()
print(f'\ntotal generated :\t{len(man):,} / {plan_df.budget.sum():,}')

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

...

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

1392654 Gomphocarpus physocarpus     + 129 ->  129/129  12.2 min

total generated :	2,356 / 2,356


## Section 4) Finalize

เขียน synthetic_manifest.csv โดย append ของ flux_zeroshot ต่อจากของ lora_boost ที่ NB03a เขียนไว้ ถ้ามี source flux_zeroshot ค้างจากการรันเก่าจะลบทิ้งก่อนแล้วเขียนใหม่ (idempotent ต่อการรัน NB03b ซ้ำ) path เก็บเป็น <species_id>/<filename> ให้ NB04 เติม prefix ของ source เองตอน fill rare

In [6]:
man = pd.read_csv(GEN_MANIFEST, dtype={'species_id': str})

new_rows = pd.DataFrame({
    'path': man.species_id + '/' + man.path,
    'species_id': man.species_id,
    'source': SOURCE,
})

SYNTH_MANIFEST = META / 'synthetic_manifest.csv'
if SYNTH_MANIFEST.exists():
    existing = pd.read_csv(SYNTH_MANIFEST, dtype={'species_id': str})
    existing = existing[existing.source != SOURCE]      # ลบ source นี้ของรอบเก่าออก กัน append ซ้ำ
    out = pd.concat([existing, new_rows], ignore_index=True)
else:
    out = new_rows

out.to_csv(SYNTH_MANIFEST, index=False)

cov = new_rows.groupby('species_id').size()
plan_df['n_synth'] = plan_df.species_id.map(cov).fillna(0).astype(int)
plan_df['ok'] = plan_df.n_synth >= plan_df.budget

print(plan_df[['species_id', 'name', 'n_train', 'budget', 'n_synth', 'ok']].to_string(index=False))
print(f'\nmanifest      :\t{SYNTH_MANIFEST}')
print(f'total rows    :\t{len(out):,}')
print(f'  lora_boost  :\t{int((out.source == "lora_boost").sum()):,}')
print(f'  flux_zeroshot:\t{int((out.source == "flux_zeroshot").sum()):,}')
print(f'all ok        :\t{plan_df.ok.all()}')

species_id                         name  n_train  budget  n_synth   ok
   1416509        Nepenthes vieillardii       20     180      180 True
   1419807           Vanilla planifolia       23     177      177 True
   1376703           Petiveria alliacea       28     172      172 True
   1409195            Pereskia aculeata       33     167      167 True
   1408557             Pongamia pinnata       36     164      164 True
   1406486             Kigelia africana       37     163      163 True
   1420787               Alocasia odora       38     162      162 True
   1408490         Pereskia grandifolia       40     160      160 True
   1385700        Barringtonia asiatica       47     153      153 True
   1356076         Tradescantia zebrina       49     151      151 True
   1397598              Cobaea scandens       49     151      151 True
   1618661 Dendrobium victoriae-reginae       51     149      149 True
   1384499          Lagenaria siceraria       56     144      144 True
   140

In [7]:
print(new_rows.head(3).to_string(index=False))
print()
sample = new_rows.iloc[0]
full = SYNTH_ROOT / sample.source / sample.path
print(f'manifest path  :\t{sample.path}')
print(f'resolves to    :\t{full}')
print(f'file exists    :\t{full.exists()}')

                            path species_id        source
1416509/1416509_flower_00000.png    1416509 flux_zeroshot
1416509/1416509_flower_00001.png    1416509 flux_zeroshot
1416509/1416509_flower_00002.png    1416509 flux_zeroshot

manifest path  :	1416509/1416509_flower_00000.png
resolves to    :	/workspace/lora-boost/data/synthetic/flux_zeroshot/1416509/1416509_flower_00000.png
file exists    :	True
